[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-10-model-serving.ipynb#scrollTo=a1f2b3c4)

---
# Day 10 · Model Serving with mlflow models serve and REST API
**certified-journeys / mlflow-certified** · Practice · Model Deployment

> **Goal for today:** Log a model with a proper signature, serve it via MLflow's built-in REST server, and send prediction requests — understanding the `/invocations` and `/ping` endpoints and how schema enforcement protects your service.

In [ ]:
%pip install -q mlflow scikit-learn pandas numpy requests

## Step 1 · What Is `mlflow models serve`?

MLflow ships a built-in model server you can launch from the CLI:

```bash
mlflow models serve -m "models:/my-model/Production" -p 5001
```

The server exposes two HTTP endpoints:

| Endpoint | Method | Purpose |
|---|---|---|
| `/ping` | GET | Health check — returns `{}` with HTTP 200 when ready |
| `/invocations` | POST | Prediction endpoint — accepts JSON or CSV payload |
| `/version` | GET | Returns MLflow version info |

In this notebook we'll spin up the server in-process using a subprocess, then call it with the `requests` library — the same pattern you'd use in a CI smoke test.

> **Colab note:** `mlflow models serve` starts a blocking Flask/gunicorn process. We launch it in a background subprocess and poll `/ping` until it responds.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# SQLite backend is required for the Model Registry (used later when loading by stage)
mlflow.set_tracking_uri("sqlite:///mlflow_serving_demo.db")
mlflow.set_experiment("day-10-model-serving")

iris = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

print("Iris dataset split:")
print(f"  Train: {X_train.shape}, Test: {X_test.shape}")
print(f"  Features: {list(iris.feature_names)}")

**What just happened?**
- We're using **SQLite** as the tracking backend because the Model Registry requires a database (not just the default file store).
- The same code works with a remote PostgreSQL backend — just swap the URI.
- We loaded the Iris dataset, which has 4 float features — a great schema-enforcement test case.

## Step 2 · Infer a Model Signature and Log the Model

A **model signature** captures the expected input and output schema. MLflow validates this schema at serving time — requests with the wrong column names or dtypes are rejected before they reach the model.

| Signature component | What it captures |
|---|---|
| `inputs` | Column names + dtypes of the features DataFrame |
| `outputs` | Dtype of the prediction (integer, float, string…) |
| `params` | Optional inference-time parameters (e.g. `threshold`) |

```python
sig = infer_signature(X_train, model.predict(X_train))
mlflow.sklearn.log_model(model, "model", signature=sig)
```

> **Always define a signature.** Without one, the server accepts any payload shape — schema errors surface inside the model rather than at the API boundary.

In [ ]:
# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)
acc = accuracy_score(y_test, clf.predict(X_test))

# Infer signature from training data and predictions
signature = infer_signature(
    X_train,                # model inputs — a DataFrame with named columns
    clf.predict(X_train),   # model outputs — integer class labels
)

print("Inferred signature:")
print("  Inputs:")
for col in signature.inputs.inputs:
    print(f"    {col.name}: {col.type}")
print(f"  Outputs: {signature.outputs.inputs[0].type}")

In [ ]:
MODEL_NAME = "iris-serving-demo"

with mlflow.start_run() as run:
    mlflow.log_params({"n_estimators": 50, "random_state": 42})
    mlflow.log_metric("accuracy", acc)

    # Log with signature AND input_example
    # input_example shows users how to format a real request payload
    mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(2),  # stored as example payload in the artifact
        registered_model_name=MODEL_NAME,  # auto-register after logging
    )
    run_id = run.info.run_id

print(f"Model logged and registered. run_id={run_id}")
print(f"Model URI: runs:/{run_id}/model")
print(f"Registered as: {MODEL_NAME} v1")
print(f"Accuracy: {acc:.4f}")

**What just happened?**
- **`infer_signature`** inspected `X_train` (a DataFrame) and the numpy array of predictions to build a typed schema automatically.
- **`input_example`** stores a sample payload alongside the model artifact — the MLflow UI renders it as a curl command you can copy-paste.
- **`registered_model_name`** in `log_model` is a one-step shortcut: it logs the artifact AND registers it to the Model Registry in a single call.

## Step 3 · Load the Saved Model and Understand the `/invocations` Payload Format

Before starting the server, let's understand the payload formats the `/invocations` endpoint accepts:

| Format | JSON key | When to use |
|---|---|---|
| `dataframe_split` | `{"columns": [...], "data": [[...]]}` | Default; explicit column names |
| `dataframe_records` | `{"dataframe_records": [{"col": val}]}` | Matches JSON records orientation |
| `instances` | `{"instances": [[...]]}` | TensorFlow Serving compatible |
| `inputs` | `{"inputs": [[...]]}` | Also TF Serving compatible |

The `dataframe_split` format is the most explicit — columns are named so schema validation can catch typos immediately.

In [ ]:
import json

# Load the model directly from the run URI (no server needed)
loaded_model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

# Build a dataframe_split payload — the format the REST server expects
sample = X_test.head(3)
payload_split = {
    "dataframe_split": {
        "columns": sample.columns.tolist(),
        "data": sample.values.tolist()
    }
}

# Build a dataframe_records payload — alternative format
payload_records = {
    "dataframe_records": sample.to_dict(orient="records")
}

print("dataframe_split payload:")
print(json.dumps(payload_split, indent=2))
print("\ndataframe_records payload:")
print(json.dumps(payload_records, indent=2))

In [ ]:
# Simulate what the /invocations endpoint does internally:
# parse the dataframe_split payload and call model.predict()

def simulate_invocations(model, payload_json: dict) -> list:
    """Mimics the MLflow /invocations handler for dataframe_split format."""
    if "dataframe_split" in payload_json:
        ds = payload_json["dataframe_split"]
        df = pd.DataFrame(data=ds["data"], columns=ds["columns"])
    elif "dataframe_records" in payload_json:
        df = pd.DataFrame(payload_json["dataframe_records"])
    else:
        raise ValueError("Unknown payload format")
    return model.predict(df).tolist()

preds_split   = simulate_invocations(loaded_model, payload_split)
preds_records = simulate_invocations(loaded_model, payload_records)

print("Predictions (dataframe_split):",   preds_split)
print("Predictions (dataframe_records):", preds_records)
print("Ground truth:                    ", y_test.head(3).tolist())
print("Both formats agree:", preds_split == preds_records)

**What just happened?**
- **`dataframe_split`** and **`dataframe_records`** both reconstruct the same DataFrame — the MLflow server handles both transparently.
- `simulate_invocations` mirrors what `mlflow.pyfunc`'s built-in Flask handler does internally — useful to understand and unit-test locally without starting a server.
- **Schema enforcement** happens before this step in the real server: if `columns` don't match the signature, MLflow returns HTTP 400 before calling `predict`.

## Step 4 · Start the MLflow Serving Server in a Background Process

In a real deployment you'd run:
```bash
mlflow models serve -m "models:/iris-serving-demo/Production" -p 5001 --no-conda
```

Key CLI flags:

| Flag | Effect |
|---|---|
| `-m` / `--model-uri` | Model URI: `runs:/`, `models:/`, or local path |
| `-p` / `--port` | Port to listen on (default 5000) |
| `--host` | Bind address (default `127.0.0.1`; use `0.0.0.0` for Docker) |
| `--no-conda` | Skip conda env creation — required in Colab |
| `--workers` | Number of gunicorn workers |

> **Production tip:** Use `--workers 4` with gunicorn for concurrent requests. The default single-worker Flask server is only for development.

In [ ]:
import subprocess
import time
import requests
import os

MODEL_URI = f"runs:/{run_id}/model"
PORT = 5001
SERVER_URL = f"http://127.0.0.1:{PORT}"

# Set the tracking URI env var so the subprocess can find the local SQLite DB
env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow_serving_demo.db"

# Launch the server as a background process
server_proc = subprocess.Popen(
    [
        "mlflow", "models", "serve",
        "-m", MODEL_URI,
        "-p", str(PORT),
        "--no-conda",        # skip conda — not available in Colab
        "--host", "127.0.0.1",
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Poll /ping until the server is ready (up to 60 seconds)
ready = False
for attempt in range(30):
    time.sleep(2)
    try:
        r = requests.get(f"{SERVER_URL}/ping", timeout=2)
        if r.status_code == 200:
            ready = True
            print(f"Server ready after {(attempt+1)*2}s — /ping returned: {r.text.strip()}")
            break
    except requests.exceptions.ConnectionError:
        pass  # server not up yet

if not ready:
    # Print stderr to diagnose startup failure
    print("Server did not start in time. stderr:")
    print(server_proc.stderr.read(2000).decode())

**What just happened?**
- **`subprocess.Popen`** launches the MLflow server non-blocking, identical to running the CLI command in a terminal tab.
- We poll **`/ping`** every 2 seconds — this is the recommended health-check endpoint used by Kubernetes readiness probes.
- **`--no-conda`** is essential in Colab (and most Docker containers) because there's no conda environment to activate.
- We pass `MLFLOW_TRACKING_URI` in the subprocess env so it finds the same SQLite DB the notebook is using.

## Step 5 · Send Prediction Requests to `/invocations`

The equivalent curl command is:
```bash
curl -d '{"dataframe_split": {"columns": [...], "data": [[...]]}}' \
     -H 'Content-Type: application/json' \
     localhost:5001/invocations
```

We'll replicate this with `requests` and also test both JSON payload formats.

In [ ]:
HEADERS = {"Content-Type": "application/json"}

# ── Request 1: dataframe_split format ─────────────────────────────────────────
response_split = requests.post(
    f"{SERVER_URL}/invocations",
    json=payload_split,
    headers=HEADERS,
)
print("dataframe_split response:")
print(f"  HTTP {response_split.status_code}")
print(f"  Body: {response_split.text}")

# ── Request 2: dataframe_records format ───────────────────────────────────────
response_records = requests.post(
    f"{SERVER_URL}/invocations",
    json=payload_records,
    headers=HEADERS,
)
print("\ndataframe_records response:")
print(f"  HTTP {response_records.status_code}")
print(f"  Body: {response_records.text}")

In [ ]:
# ── Request 3: Deliberately wrong column name — test schema enforcement ────────
bad_payload = {
    "dataframe_split": {
        "columns": ["wrong_col_1", "wrong_col_2", "wrong_col_3", "wrong_col_4"],
        "data": [[5.1, 3.5, 1.4, 0.2]]
    }
}

response_bad = requests.post(
    f"{SERVER_URL}/invocations",
    json=bad_payload,
    headers=HEADERS,
)
print("Schema violation response:")
print(f"  HTTP {response_bad.status_code}")
print(f"  Body: {response_bad.text[:300]}")  # truncate long error messages

print("\nSchema enforcement blocked the bad request:", response_bad.status_code == 400)

**What just happened?**
- **HTTP 200** from `/invocations` returns a JSON object with a `predictions` key containing the model output.
- **Schema enforcement** on the bad payload returns **HTTP 400** — the request never reaches the model. This is the key benefit of defining a model signature.
- **Both payload formats** produce identical predictions — choose `dataframe_split` when column names matter (always in production).

## Step 6 · Inspect the Saved Model Artifact Structure

Every MLflow model artifact follows a standard directory layout regardless of the model flavor:

```
model/
  MLmodel          ← YAML manifest: flavors, signature, input_example path
  model.pkl        ← serialized model (sklearn flavor)
  conda.yaml       ← conda environment spec
  requirements.txt ← pip requirements
  input_example.json ← the example we passed at log_model time
```

The `MLmodel` file is the key — the serving infrastructure reads it to know which flavor loader to use and what the expected schema is.

In [ ]:
import yaml
import os
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Download the model artifact to inspect its MLmodel file
local_path = client.download_artifacts(run_id, "model", dst_path="/tmp/serving_demo")
mlmodel_path = os.path.join(local_path, "MLmodel")

with open(mlmodel_path) as f:
    mlmodel = yaml.safe_load(f)

print("MLmodel flavors:")
for flavor_name, flavor_info in mlmodel.get("flavors", {}).items():
    print(f"  {flavor_name}: {list(flavor_info.keys())}")

print("\nSignature inputs:")
for col in mlmodel.get("signature", {}).get("inputs", []):
    print(f"  {col}")

print("\nSignature outputs:")
print(f"  {mlmodel.get('signature', {}).get('outputs', 'none')}")

print("\nArtifact files:")
for fname in sorted(os.listdir(local_path)):
    fsize = os.path.getsize(os.path.join(local_path, fname))
    print(f"  {fname} ({fsize:,} bytes)")

**What just happened?**
- **`MLmodel`** is a YAML manifest that makes every MLflow model self-describing — the serving layer reads it to determine which Python loader to call.
- **Two flavors** are always saved for sklearn models: `sklearn` (native) and `python_function` (generic pyfunc interface). The serving server uses `pyfunc`.
- **`signature`** is embedded directly in the MLmodel manifest — this is what the server reads at startup to configure schema validation.
- **`input_example.json`** stores the sample payload we passed; the UI renders it as a ready-to-use curl snippet.

## Step 7 · Promote to Production and Serve by Registry Stage

In real deployments the model URI in `mlflow models serve` points to the Registry, not a specific run:

```bash
# Serve always-latest Production version — no URI change when you promote a new model
mlflow models serve -m "models:/iris-serving-demo/Production" -p 5001 --no-conda

# Or use an alias (MLflow 2.x preferred)
mlflow models serve -m "models:/iris-serving-demo@champion" -p 5001 --no-conda
```

This decouples deployment from model promotion — your serving command never changes, only the Registry pointer changes.

In [ ]:
# Promote the registered model to Production and set the 'champion' alias
client.transition_model_version_stage(
    name=MODEL_NAME,
    version="1",
    stage="Production",
    archive_existing_versions=False,
)

# Set alias for MLflow 2.x alias-based serving
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version="1",
)

# Verify: load the model using the registry stage URI
prod_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Production")
alias_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

prod_preds  = prod_model.predict(X_test).tolist()
alias_preds = alias_model.predict(X_test).tolist()

print(f"Production URI → model v1, accuracy: {accuracy_score(y_test, prod_preds):.4f}")
print(f"Alias URI      → model v1, accuracy: {accuracy_score(y_test, alias_preds):.4f}")
print(f"Stage and alias predictions match: {prod_preds == alias_preds}")

print("\nThe CLI equivalent serving commands would be:")
print(f'  mlflow models serve -m "models:/{MODEL_NAME}/Production" -p 5002 --no-conda')
print(f'  mlflow models serve -m "models:/{MODEL_NAME}@champion"  -p 5002 --no-conda')

**What just happened?**
- **`models:/<name>/Production`** and **`models:/<name>@champion`** both resolve to version 1 — confirming both URI styles work.
- **The serving command doesn't change** when you promote a new model version — only the Registry pointer changes. This is why using Registry URIs (not `runs:/`) in production is essential.
- **Alias-based URIs** are the MLflow 2.x recommended approach because stage-based promotion is being deprecated.

In [ ]:
# Clean up: terminate the background server process
if 'server_proc' in dir() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait(timeout=5)
    print("Server process terminated.")
else:
    print("Server was not running or already stopped.")

# Final summary of what we accomplished
print("\nDay 10 serving demo complete:")
print(f"  Model: {MODEL_NAME}")
print(f"  Signature: {len(signature.inputs.inputs)} input columns, 1 output")
print(f"  Server endpoints tested: /ping, /invocations (split + records + bad payload)")
print(f"  Schema enforcement: blocked bad column names with HTTP 400")
print(f"  Registry: promoted to Production, set champion alias")

In [ ]:
# Challenge: Build a serving smoke-test function
#
# Write a function `smoke_test_server(base_url, sample_df)` that:
#   1. Hits /ping and asserts HTTP 200
#   2. Builds a dataframe_split payload from sample_df
#   3. Posts to /invocations and asserts HTTP 200
#   4. Parses the response JSON and returns the list of predictions
#   5. Raises AssertionError with a helpful message if any check fails
#
# Bonus: also test a malformed payload and assert HTTP 4xx
#
# Hint: response.json() on /invocations returns {"predictions": [...]}

def smoke_test_server(base_url: str, sample_df) -> list:
    # Your solution here
    pass

# Test it (uncomment after implementing — requires server_proc to be running):
# preds = smoke_test_server(SERVER_URL, X_test.head(5))
# print("Smoke test predictions:", preds)

---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| `mlflow models serve` | CLI command that starts a Flask/gunicorn REST server for any MLflow model |
| `/ping` | Health-check endpoint — returns `{}` with HTTP 200 when server is ready |
| `/invocations` | Prediction endpoint — accepts `dataframe_split`, `dataframe_records`, or TF Serving formats |
| `infer_signature(X, y_pred)` | Automatically infers input/output schema from data; stored in the MLmodel manifest |
| Schema enforcement | Requests with wrong column names or types are rejected with HTTP 400 before reaching the model |
| `models:/<name>/Production` | Serve the Production-stage version — no URI change needed when promoting a new model |
| `models:/<name>@champion` | Alias-based URI — MLflow 2.x preferred; decouples serving from stage deprecation |
| `--no-conda` | Required when conda is unavailable (Colab, Docker, CI) |

> **Tip:** Always define a model signature — it documents expected inputs/outputs and enables automatic schema validation when the model is served.

---
## What's next
**Day 11** → Cloud Backends — configuring S3, Azure Blob, and GCS as MLflow artifact stores, and connecting a remote PostgreSQL tracking server.

Mark Day 10 complete in your [tracker](../index.html).